# ?? Jazz AI V1.0 ? ~0.6B Parameter Hindi/Hinglish LLM From Scratch

**Kaggle-ready copy** of the Colab notebook.

Before running: Settings ? Accelerator ? GPU T4 x2 or GPU P100.

## Pipeline
1. ? Check GPU & VRAM
2. ? Install dependencies
3. ? Use Kaggle working/checkpoint directory
4. ? Write all source files
5. ? Write SFT training data
6. ?? Prepare Hindi Wikipedia data + train tokenizer
7. ?? Pretrain model with checkpoints
8. ?? SFT fine-tuning
9. ?? Interactive chat


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU')

In [ ]:
!pip install -q regex datasets tqdm tokenizers
import os
os.makedirs('jazz_ai', exist_ok=True)
os.makedirs('jazz_ai/data', exist_ok=True)
os.makedirs('jazz_ai/out/sft', exist_ok=True)
%cd jazz_ai
print('Working dir:', os.getcwd())

In [ ]:
import os
DRIVE_DIR = '/kaggle/working/jazz_ai_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Kaggle checkpoints will be saved to:', DRIVE_DIR)


## Write Source Files

In [ ]:
import os
if os.path.exists('config.py'):
    os.remove('config.py')
    print('Deleted old config.py')

config_py = '''
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class JazzConfig:
    name: str = "jazz-tiny-cpu"
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_kv_head: Optional[int] = None
    n_embd: int = 384
    ffn_mult: float = 8 / 3
    ffn_multiple_of: int = 64
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    tie_weights: bool = True
    dropout: float = 0.0
    batch_size: int = 16
    grad_accum_steps: int = 1
    max_iters: int = 3000
    eval_interval: int = 250
    eval_iters: int = 50
    log_interval: int = 25
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    warmup_iters: int = 100
    lr_decay_iters: int = 3000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    device: str = "cpu"
    dtype: str = "float32"
    compile: bool = False
    seed: int = 1337
    data_dir: str = "data"
    out_dir: str = "out"
    dataset: str = "tinyshakespeare"
    gradient_checkpointing: bool = False

    def param_count_estimate(self) -> int:
        h = self.n_embd
        ffn_hidden = int(self.ffn_mult * h)
        m = self.ffn_multiple_of
        ffn_hidden = m * ((ffn_hidden + m - 1) // m)
        per_layer = 4 * h * h + 3 * h * ffn_hidden
        embed = self.vocab_size * h * (1 if self.tie_weights else 2)
        return self.n_layer * per_layer + embed

PRESETS = {
    "tiny-cpu": JazzConfig(),
    "jazz-500m": JazzConfig(
        name="jazz-500m",
        vocab_size=32000, block_size=1024,
        n_layer=24, n_head=16, n_kv_head=8, n_embd=1280,
        ffn_mult=8/3, ffn_multiple_of=64,
        batch_size=8, grad_accum_steps=4,
        max_iters=100000, lr_decay_iters=100000,
        warmup_iters=2000, learning_rate=3e-4, min_lr=3e-5,
        weight_decay=0.1, dataset="hinglish",
        device="cuda", dtype="bfloat16", compile=True,
        eval_interval=500, eval_iters=50, log_interval=10,
    ),
    "jazz-1b": JazzConfig(
        name="jazz-1b",
        vocab_size=8000,
        block_size=1024,
        n_layer=24,
        n_head=12,
        n_kv_head=4,
        n_embd=1536,
        ffn_mult=8/3,
        ffn_multiple_of=64,
        batch_size=2,
        grad_accum_steps=8,
        max_iters=50000,
        lr_decay_iters=50000,
        warmup_iters=1000,
        learning_rate=3e-4,
        min_lr=3e-5,
        weight_decay=0.1,
        dataset="hinglish",
        device="cuda",
        dtype="float16",
        compile=False,
        gradient_checkpointing=True,
        eval_interval=500,
        eval_iters=50,
        log_interval=10,
    ),
}

def get_config(preset: str = "tiny-cpu", **overrides) -> JazzConfig:
    if preset not in PRESETS:
        raise ValueError(f"unknown preset {preset!r}; choose from {list(PRESETS)}")
    base = PRESETS[preset]
    cfg = JazzConfig(**{**asdict(base), **overrides})
    return cfg
'''
with open('config.py', 'w') as f:
    f.write(config_py.strip())

# Verify
from importlib import reload
import config
reload(config)
from config import get_config
cfg = get_config('jazz-1b')
print(f'✅ config.py OK — {cfg.name}')
print(f'   block_size={cfg.block_size}, grad_accum={cfg.grad_accum_steps}, max_iters={cfg.max_iters:,}')
print(f'   Effective batch: {cfg.batch_size * cfg.grad_accum_steps} | Tokens/iter: {cfg.batch_size * cfg.block_size * cfg.grad_accum_steps:,}')

In [ ]:
model_py = open('/dev/stdin').read() if False else r"""
from __future__ import annotations
import math
from dataclasses import dataclass
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import JazzConfig

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm.to(dtype) * self.weight

def precompute_rope_cache(head_dim, max_seq, base, device=None):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    rx1 = x1 * cos - x2 * sin
    rx2 = x1 * sin + x2 * cos
    return torch.stack((rx1, rx2), dim=-1).flatten(-2).type_as(x)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head or cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.wq = nn.Linear(cfg.n_embd, self.n_head * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_head * self.head_dim, cfg.n_embd, bias=False)
        self.dropout = cfg.dropout
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.wo(y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hidden = int(cfg.ffn_mult * cfg.n_embd)
        m = cfg.ffn_multiple_of
        hidden = m * ((hidden + m - 1) // m)
        self.w1 = nn.Linear(cfg.n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, cfg.n_embd, bias=False)
        self.w3 = nn.Linear(cfg.n_embd, hidden, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)
        self.gradient_checkpointing = getattr(cfg, 'gradient_checkpointing', False)

    def _attn_fn(self, x, cos, sin):
        return self.attn(self.attn_norm(x), cos, sin)

    def _ffn_fn(self, x):
        return self.ffn(self.ffn_norm(x))

    def forward(self, x, cos, sin):
        if self.training and self.gradient_checkpointing:
            x = x + torch.utils.checkpoint.checkpoint(self._attn_fn, x, cos, sin, use_reentrant=False)
            x = x + torch.utils.checkpoint.checkpoint(self._ffn_fn, x, use_reentrant=False)
        else:
            x = x + self.attn(self.attn_norm(x), cos, sin)
            x = x + self.ffn(self.ffn_norm(x))
        return x

class JazzLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        if cfg.tie_weights:
            self.lm_head.weight = self.tok_emb.weight
        head_dim = cfg.n_embd // cfg.n_head
        cos, sin = precompute_rope_cache(head_dim, cfg.block_size, cfg.rope_base)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('wo.weight') or pn.endswith('w2.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def resize_token_embeddings(self, new_vocab_size):
        old = self.tok_emb.weight.data
        old_n, dim = old.shape
        if new_vocab_size == old_n: return
        new_emb = nn.Embedding(new_vocab_size, dim)
        nn.init.normal_(new_emb.weight, mean=0.0, std=0.02)
        n_copy = min(old_n, new_vocab_size)
        with torch.no_grad():
            new_emb.weight[:n_copy].copy_(old[:n_copy])
        new_emb.weight.data = new_emb.weight.data.to(old.dtype).to(old.device)
        self.tok_emb = new_emb
        self.lm_head = nn.Linear(dim, new_vocab_size, bias=False).to(old.device)
        if self.cfg.tie_weights:
            self.lm_head.weight = self.tok_emb.weight
        self.cfg.vocab_size = new_vocab_size
    def num_parameters(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok_emb.weight.numel()
            if not self.cfg.tie_weights: n -= self.lm_head.weight.numel()
        return n
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos, sin = self.rope_cos, self.rope_sin
        for block in self.blocks:
            x = block(x, cos, sin)
        x = self.norm(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        return self.lm_head(x[:, -1:, :]), None
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, nxt), dim=1)
            if eos_token_id is not None and (nxt == eos_token_id).all():
                break
        return idx
"""
with open('model.py', 'w') as f:
    f.write(model_py.strip())
print('model.py written')

In [ ]:
tokenizer_py = r"""
from __future__ import annotations
import json, os, tempfile
from typing import Dict, List, Optional
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SPECIAL_TOKENS = ['<|bos|>', '<|eos|>', '<|pad|>', '<|unk|>', '<|im_start|>']

class BPETokenizer:
    def __init__(self):
        self.tokenizer: Optional[Tokenizer] = None
        self.special_tokens: Dict[str, int] = {}

    @property
    def vocab_size(self) -> int:
        return self.tokenizer.get_vocab_size() if self.tokenizer else 0

    @property
    def eos_token_id(self) -> int:
        return self.special_tokens.get('<|eos|>', 1)

    def train(self, text: str, vocab_size: int, verbose: bool = True):
        tokenizer = Tokenizer(models.BPE(unk_token="<|unk|>"))
        tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tokenizer.decoder = decoders.ByteLevel()

        trainer = trainers.BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=SPECIAL_TOKENS,
            show_progress=verbose,
        )

        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt', encoding='utf-8') as f:
            f.write(text)
            tmp_path = f.name

        try:
            tokenizer.train([tmp_path], trainer)
        finally:
            os.unlink(tmp_path)

        self.tokenizer = tokenizer
        vocab = tokenizer.get_vocab()
        self.special_tokens = {tok: vocab[tok] for tok in SPECIAL_TOKENS if tok in vocab}
        if verbose:
            print(f'  BPE trained: vocab={self.vocab_size}', flush=True)

    def encode(self, text: str) -> List[int]:
        if self.tokenizer is None:
            raise RuntimeError("Tokenizer not trained or loaded")
        return self.tokenizer.encode(text).ids

    def decode(self, ids: List[int]) -> str:
        if self.tokenizer is None:
            raise RuntimeError("Tokenizer not trained or loaded")
        return self.tokenizer.decode(ids)

    def add_special_tokens(self, tokens: List[str]) -> List[int]:
        vocab = self.tokenizer.get_vocab() if self.tokenizer else {}
        new_ids = []
        for tok in tokens:
            if tok in vocab:
                new_ids.append(vocab[tok])
        return new_ids

    def apply_chat_template(self, messages, add_generation_prompt=True, return_assistant_mask=False):
        im_start = self.special_tokens.get('<|im_start|>', self.special_tokens.get('<|bos|>', 0))
        im_end = self.eos_token_id
        ids, mask = [], []
        for msg in messages:
            role = msg['role']
            content = msg['content']
            header = self.encode(f'\n{role}\n')
            body = self.encode(content)
            is_asst = (role == 'assistant')
            ids += [im_start] + header + body + [im_end] + self.encode('\n')
            mask += ([0] + [0]*len(header) +
                     ([1]*len(body) if is_asst else [0]*len(body)) +
                     [0] + [0])
        if add_generation_prompt:
            prompt = self.encode('\nassistant\n')
            ids += [im_start] + prompt
            mask += [0] + [0]*len(prompt)
        if return_assistant_mask:
            return ids, mask
        return ids

    def save(self, path: str):
        if self.tokenizer is None:
            raise RuntimeError("Tokenizer not trained or loaded")
        self.tokenizer.save(path)
        meta = {'special_tokens': self.special_tokens, 'vocab_size': self.vocab_size}
        with open(path + '.meta', 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False)

    @classmethod
    def load(cls, path: str) -> 'BPETokenizer':
        t = cls()
        t.tokenizer = Tokenizer.from_file(path)
        meta_path = path + '.meta'
        if os.path.exists(meta_path):
            with open(meta_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
            t.special_tokens = meta.get('special_tokens', {})
        else:
            vocab = t.tokenizer.get_vocab()
            t.special_tokens = {tok: vocab[tok] for tok in SPECIAL_TOKENS if tok in vocab}
        return t
"""
with open('tokenizer.py', 'w') as f:
    f.write(tokenizer_py.strip())
print('tokenizer.py written (fast HF tokenizer)')

In [ ]:
prepare_data_py = r"""
import argparse, os, json, time
import numpy as np
from tokenizer import BPETokenizer, SPECIAL_TOKENS
from config import get_config

def _load_hinglish(max_bytes=300*1024*1024):
    from datasets import load_dataset
    print('Loading Hindi Wikipedia...', flush=True)
    ds = load_dataset('wikimedia/wikipedia', '20231101.hi', split='train', streaming=True, trust_remote_code=True)
    text, total = [], 0
    for ex in ds:
        t = ex.get('text', '').strip()
        if not t: continue
        text.append(t)
        total += len(t.encode('utf-8'))
        if total >= max_bytes:
            break
        if len(text) % 1000 == 0:
            print(f'  loaded {len(text)} articles, {total/1e6:.1f} MB', flush=True)
    return '\n\n'.join(text)

def _encode_chunked(tok, text, chunk_mb=5):
    chunk_size = chunk_mb * 1024 * 1024
    all_ids = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        all_ids.extend(tok.encode(chunk))
        if (i // chunk_size + 1) % 10 == 0:
            print(f'  encoded {(i+len(chunk))/1e6:.1f} MB...', flush=True)
    return all_ids

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--preset', default='jazz-500m')
    ap.add_argument('--max_train_mb', type=int, default=500)
    ap.add_argument('--bpe_train_mb', type=int, default=100)
    ap.add_argument('--vocab_size', type=int, default=None)
    args = ap.parse_args()
    cfg = get_config(args.preset)
    vocab_size = args.vocab_size or cfg.vocab_size
    data_dir = cfg.data_dir
    os.makedirs(data_dir, exist_ok=True)
    tok_path = os.path.join(data_dir, 'tokenizer.json')

    print(f'Loading text (max {args.max_train_mb} MB)...', flush=True)
    text = _load_hinglish(max_bytes=args.max_train_mb*1024*1024)
    print(f'Total text: {len(text)/1e6:.1f} MB', flush=True)

    if os.path.exists(tok_path):
        print(f'Reusing existing tokenizer at {tok_path}', flush=True)
        tok = BPETokenizer.load(tok_path)
    else:
        bpe_bytes = args.bpe_train_mb * 1024 * 1024
        bpe_text = text[:bpe_bytes]
        print(f'Training BPE on {len(bpe_text)/1e6:.1f} MB, vocab={vocab_size}...', flush=True)
        tok = BPETokenizer()
        tok.train(bpe_text, vocab_size=vocab_size, verbose=True)
        tok.save(tok_path)
        print(f'Tokenizer saved to {tok_path} (vocab={tok.vocab_size})', flush=True)

    val_split = min(0.05, 50000 / max(1, len(text)))
    val_bytes = int(len(text) * val_split)
    val_text   = text[-val_bytes:]
    train_text = text[:-val_bytes]

    for split, t in [('train', train_text), ('val', val_text)]:
        out_path = os.path.join(data_dir, f'{split}.bin')
        print(f'Encoding {split} ({len(t)/1e6:.1f} MB)...', flush=True)
        ids = _encode_chunked(tok, t, chunk_mb=10)
        arr = np.array(ids, dtype=np.uint16)
        arr.tofile(out_path)
        print(f'  {split}: {len(arr):,} tokens saved to {out_path}', flush=True)

if __name__ == '__main__':
    main()
"""
with open('prepare_data.py', 'w') as f:
    f.write(prepare_data_py.strip())
print('prepare_data.py written')

In [ ]:
import shutil, os

train_py = open('/dev/stdin').read() if False else r"""
from __future__ import annotations
import argparse, math, os, time, shutil
from contextlib import nullcontext
from dataclasses import asdict
import numpy as np
import torch
from config import get_config, JazzConfig
from model import JazzLM
from tokenizer import BPETokenizer

def get_batch(split, data_dir, block_size, batch_size, device):
    path = os.path.join(data_dir, f'{split}.bin')
    data = np.memmap(path, dtype=np.uint16, mode='r')
    ix = np.random.randint(0, len(data) - block_size - 1, size=(batch_size,))
    x = np.stack([data[i:i+block_size].astype(np.int64) for i in ix])
    y = np.stack([data[i+1:i+1+block_size].astype(np.int64) for i in ix])
    x, y = torch.from_numpy(x), torch.from_numpy(y)
    if device.startswith('cuda'):
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

def cosine_lr(it, cfg):
    if it < cfg.warmup_iters: return cfg.learning_rate * (it+1) / max(1, cfg.warmup_iters)
    if it > cfg.lr_decay_iters: return cfg.min_lr
    ratio = (it - cfg.warmup_iters) / max(1, cfg.lr_decay_iters - cfg.warmup_iters)
    return cfg.min_lr + 0.5*(1.0+math.cos(math.pi*ratio))*(cfg.learning_rate - cfg.min_lr)

@torch.no_grad()
def estimate_loss(model, cfg):
    out = {}
    model.eval()
    for split in ('train', 'val'):
        losses = []
        for _ in range(cfg.eval_iters):
            x, y = get_batch(split, cfg.data_dir, cfg.block_size, cfg.batch_size, cfg.device)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses)/len(losses)
    model.train()
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--preset', default='jazz-500m')
    ap.add_argument('--max_iters', type=int, default=None)
    ap.add_argument('--batch_size', type=int, default=None)
    ap.add_argument('--device', default=None)
    ap.add_argument('--resume', action='store_true')
    ap.add_argument('--drive_dir', default=None, help='Google Drive path to sync checkpoints')
    args = ap.parse_args()
    overrides = {k: getattr(args,k) for k in ('max_iters','batch_size','device') if getattr(args,k) is not None}
    cfg = get_config(args.preset, **overrides)
    tok_path = os.path.join(cfg.data_dir, 'tokenizer.json')
    if os.path.exists(tok_path):
        tok = BPETokenizer.load(tok_path)
        cfg.vocab_size = tok.vocab_size
        print(f'tokenizer: vocab={cfg.vocab_size}')
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    if cfg.device.startswith('cuda') and not torch.cuda.is_available():
        cfg.device = 'cpu'
    device = cfg.device
    os.makedirs(cfg.out_dir, exist_ok=True)
    if cfg.dtype == 'bfloat16' and device.startswith('cuda'):
        amp_dtype = torch.bfloat16
    elif cfg.dtype == 'float16' and device.startswith('cuda'):
        amp_dtype = torch.float16
    else:
        amp_dtype = torch.float32
    amp_ctx = (torch.amp.autocast(device_type='cuda', dtype=amp_dtype)
               if device.startswith('cuda') and amp_dtype != torch.float32 else nullcontext())
    scaler = torch.amp.GradScaler('cuda') if amp_dtype == torch.float16 else None
    model = JazzLM(cfg).to(device)
    print(f'params: {model.num_parameters():,}')
    if cfg.gradient_checkpointing:
        print('gradient checkpointing: enabled')
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (decay if p.dim() >= 2 else no_decay).append(p)
    optimizer = torch.optim.AdamW(
        [{'params': decay, 'weight_decay': cfg.weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2))
    ckpt_path = os.path.join(cfg.out_dir, 'ckpt.pt')
    start_iter, best_val = 0, float('inf')
    if args.resume and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        start_iter = ck.get('iter', 0) + 1
        best_val = ck.get('best_val', float('inf'))
        print(f'resumed from iter {start_iter}')
    if cfg.compile and hasattr(torch, 'compile'):
        print('torch.compile()...')
        model = torch.compile(model)
    model.train()
    t0 = time.time()
    for it in range(start_iter, cfg.max_iters):
        lr = cosine_lr(it, cfg)
        for pg in optimizer.param_groups: pg['lr'] = lr
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0
        for _ in range(cfg.grad_accum_steps):
            x, y = get_batch('train', cfg.data_dir, cfg.block_size, cfg.batch_size, device)
            with amp_ctx:
                _, loss = model(x, y)
                loss = loss / cfg.grad_accum_steps
            if scaler: scaler.scale(loss).backward()
            else: loss.backward()
            loss_accum += loss.item()
        if cfg.grad_clip > 0:
            if scaler: scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        if scaler: scaler.step(optimizer); scaler.update()
        else: optimizer.step()
        if it % cfg.log_interval == 0:
            dt = time.time() - t0; t0 = time.time()
            tps = cfg.batch_size*cfg.block_size*cfg.grad_accum_steps / max(dt/cfg.log_interval, 1e-9)
            print(f'iter {it:6d} | loss {loss_accum:.4f} | lr {lr:.2e} | {dt*1000/max(1,cfg.log_interval):.1f}ms | ~{tps:.0f}tok/s', flush=True)
        if it > 0 and it % cfg.eval_interval == 0:
            losses = estimate_loss(model, cfg)
            print(f'  eval@{it}: train={losses["train"]:.4f} val={losses["val"]:.4f}', flush=True)
            if losses['val'] < best_val:
                best_val = losses['val']
                m = model._orig_mod if hasattr(model, '_orig_mod') else model
                torch.save({'model': m.state_dict(), 'optimizer': optimizer.state_dict(),
                            'config': asdict(cfg), 'iter': it, 'best_val': best_val}, ckpt_path)
                print(f'  saved {ckpt_path}', flush=True)
                if args.drive_dir:
                    shutil.copy(ckpt_path, os.path.join(args.drive_dir, 'ckpt.pt'))
                    print(f'  synced to Drive', flush=True)
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    torch.save({'model': m.state_dict(), 'optimizer': optimizer.state_dict(),
                'config': asdict(cfg), 'iter': cfg.max_iters-1}, os.path.join(cfg.out_dir, 'ckpt_final.pt'))
    if args.drive_dir:
        shutil.copy(os.path.join(cfg.out_dir,'ckpt_final.pt'), os.path.join(args.drive_dir,'ckpt_final.pt'))
    print('training complete.')

if __name__ == '__main__':
    main()
"""
with open('train.py', 'w') as f:
    f.write(train_py.strip())
print('train.py written')

In [ ]:
import json

sft_data = [
    # --- General assistant ---
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a helpful Hindi/Hinglish assistant."}, {"role": "user", "content": "Tum kaun ho?"}, {"role": "assistant", "content": "Main Jazz AI V1.0 hoon — ek from-scratch Hindi/Hinglish language model. Koi bhi sawaal poochho!"}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Aapka naam kya hai?"}, {"role": "assistant", "content": "Mera naam Jazz AI V1.0 hai. Main ek Hindi/Hinglish chat assistant hoon."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "India ki rajdhani kya hai?"}, {"role": "assistant", "content": "India ki rajdhani New Delhi hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Python kya hai?"}, {"role": "assistant", "content": "Python ek popular programming language hai jo simple syntax ke liye jaani jaati hai. Data science, AI aur web development mein bahut use hoti hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Machine learning kya hota hai?"}, {"role": "assistant", "content": "Machine learning AI ka ek part hai jisme computer data se khud seekhta hai bina explicitly program kiye."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Hello!"}, {"role": "assistant", "content": "Hello! Main Jazz AI hoon. Aap kaise ho? Kuch help chahiye?"}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "2 + 2 kitna hota hai?"}, {"role": "assistant", "content": "2 + 2 = 4!"}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Deep learning kya hai?"}, {"role": "assistant", "content": "Deep learning neural networks ka use karke patterns seekhne ka tarika hai. Image recognition, NLP aur AI mein use hota hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Mujhe motivate karo, exam hai kal."}, {"role": "assistant", "content": "Bhai, jo padha hai usse revise karo. Thoda so bhi jao — neend ke bina dimaag kaam nahi karta. Tum kar loge, confidence rakho."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0."}, {"role": "user", "content": "Kya tum pretrained ho?"}, {"role": "assistant", "content": "Nahi. Mujhe pretrained weights se initialize nahi kiya gaya — sab weights from scratch Hindi Wikipedia pe train kiye gaye hain. Architecture: RoPE, RMSNorm, SwiGLU, GQA."}]},
    
    # --- Dating coach ---
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach. Give ready-to-send Hindi/Hinglish lines first, then a one-line reason. Be bold, playful, concise. Never pressure after rejection."}, {"role": "user", "content": "Uski profile pe uski smile bahut achhi hai, kya likhu pehle message mein?"}, {"role": "assistant", "content": "Try karo: 'Teri smile dekh ke lag raha hai tu sweet dikhti hai, par thodi trouble bhi hai. Sach bata.'\n\nKyun: Curiosity + light assumption — boring 'hi' se zyada interesting, reply karna easy hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach. Give ready-to-send lines first, then a one-line reason."}, {"role": "user", "content": "Usne 'haha nice' reply kiya, ab kya bolun?"}, {"role": "assistant", "content": "Bol do: 'Nice matlab impressed ho ya politely judge kar rahi ho?'\n\nKyun: Uska interest test karta hai aur conversation aage badhti hai bina needy lage."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Woh ek traveler hai, kya interesting opener hoga?"}, {"role": "assistant", "content": "Try: 'Wait, tu woh type lagti hai jise trip mein plan se zyada unplanned cheezein pasand aati hain. Sach hai?'\n\nKyun: Profile se assumption banaya — feels personal, easy yes/no reply create karta hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Usne bol diya interested nahi hai. Ab kya karu?"}, {"role": "assistant", "content": "Respectfully close karo: 'All good, thanks for being honest. Take care.'\n\nPhir move on. Pressure mat karo, follow-up chase mat karo. Yahi mature aur confident response hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Pehla message bhej diya par reply nahi aaya, follow up karu?"}, {"role": "assistant", "content": "Ek baar follow-up theek hai: 'Busy lag rahi ho — koi baat nahi, jab time ho batana.'\n\nUske baad wait karo. Double text mat karo. Agar phir bhi no reply — move on gracefully."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Date pe kaisa behave karun? Nervous hoon."}, {"role": "assistant", "content": "3 simple cheezein:\n1. Curiosity rakho — seedha interview mat karo, light tease aur assumptions karo.\n2. Present raho — phone side mein rakho.\n3. Comfortable silence se daro mat — sign hai ki dono relax ho.\n\nNervousness normal hai — bas present raho."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Usse kaise pata chalega ki mujhe usmein interest hai bina directly bolun?"}, {"role": "assistant", "content": "Subtle signals jo kaam karte hain:\n- Uski baaton ko yaad rakho aur baad mein reference karo.\n- Light teasing karo — sirf unhe karte hain jinhe actually like karte ho.\n- Attention dete waqt phone side mein rakho.\n- Ek playful assumption banao uske baare mein."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Woh bahut kam reply karti hai, iska matlab kya hai?"}, {"role": "assistant", "content": "Do possibilities:\n1. Busy hai ya texting uska style nahi.\n2. Interest low hai.\n\nEk engaging question poochho. Agar phir bhi no-effort reply — interest nahi hai. Respectfully disengage karo, apna time waste mat karo."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Conversation boring ho gayi hai, kaise interesting banau?"}, {"role": "assistant", "content": "Topic shift karo with an assumption: 'Tujhe lagta hai tu zyada adventurous hai ya comfortable life prefer karti hai?'\n\nYeh imagination game shuru karta hai — boring Q&A se bahar nikalta hai."}]},
    {"messages": [{"role": "system", "content": "You are Jazz AI V1.0, a dating coach."}, {"role": "user", "content": "Woh busy hai aur keh rahi hai baad mein baat karein, kya karu?"}, {"role": "assistant", "content": "'Sure, no rush.' — bas itna.\n\nAvailable rehna okay hai, desperate dikhna nahi. Usse space do, woh khud aayegi agar interested hai."}]},
]

os.makedirs('data', exist_ok=True)
with open('data/sft_seed.jsonl', 'w', encoding='utf-8') as f:
    for row in sft_data:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'SFT data: {len(sft_data)} conversations written to data/sft_seed.jsonl')
print(f'  - {sum(1 for d in sft_data if "dating coach" in d["messages"][0]["content"])} dating coach conversations')
print(f'  - {sum(1 for d in sft_data if "dating coach" not in d["messages"][0]["content"])} general assistant conversations')

## Step 1: Prepare Hindi Data + Train BPE Tokenizer (8K vocab)

In [ ]:
!python prepare_data.py --preset jazz-1b --max_train_mb 800 --bpe_train_mb 10 --vocab_size 8000

## Step 2: Model & VRAM Check

In [ ]:
import torch
from config import get_config

cfg = get_config('jazz-1b')
p = cfg.param_count_estimate()
print(f'Model      : {cfg.name}')
print(f'Params     : {p/1e9:.2f}B ({p:,})')
print(f'Layers     : {cfg.n_layer},  Embd: {cfg.n_embd},  Heads: {cfg.n_head} (KV: {cfg.n_kv_head})')
print(f'Block size : {cfg.block_size},  Vocab: {cfg.vocab_size}')
print(f'Batch      : {cfg.batch_size} x GradAccum: {cfg.grad_accum_steps} = eff {cfg.batch_size*cfg.grad_accum_steps}')
print(f'Max iters  : {cfg.max_iters:,}')
print(f'Dtype      : {cfg.dtype}')
print(f'Compile    : {cfg.compile}')
print(f'GradCkpt   : {cfg.gradient_checkpointing}')

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    est_gb = p * 2 / 1e9 + p * 8 / 1e9  # float16 weights + fp32 optimizer
    print(f'\nGPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {vram:.1f} GB')
    print(f'Est. memory: {est_gb:.1f} GB (weights + optimizer)')
    if est_gb < vram * 0.85:
        print('Status     : OK — fits in VRAM ✅')
    else:
        print('WARNING    : May OOM — reduce batch_size to 1 and grad_accum_steps to 32')
else:
    print('\nWARNING: No GPU! Go to Runtime → Change runtime type → GPU')

### Kaggle runtime note
This full pretraining run is long. Kaggle free GPU sessions can stop before completion, so use the resume cell and keep checkpoint outputs saved.


In [ ]:
# ? START PRETRAINING ? long run; checkpoints save to Kaggle working dir
!python train.py --preset jazz-1b --drive_dir /kaggle/working/jazz_ai_checkpoints


In [ ]:
# ? RESUME CELL ? run this after session restart if checkpoint exists
!python train.py --preset jazz-1b --resume --drive_dir /kaggle/working/jazz_ai_checkpoints


In [ ]:
# (removed — duplicate cell)

## Step 3: SFT Fine-Tuning (ChatML Instruct Format)

In [ ]:
# Kaggle-ready: SFT tokenizer setup is handled by finetune_sft.py.
print('SFT helper ready.')


In [ ]:
# ? RUN SFT ? fine-tunes the pretrained base on the ChatML instruct data
!python finetune_sft.py --preset jazz-1b --max_iters 2000 --batch_size 2 \
    --drive_dir /kaggle/working/jazz_ai_checkpoints


## Step 4: Chat with Jazz AI V1.0

In [ ]:
import torch
from config import get_config, JazzConfig
from model import JazzLM
from tokenizer import BPETokenizer

CKPT = 'out/sft/sft_ckpt.pt'
TOK  = 'data/tokenizer.json'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tok = BPETokenizer.load(TOK)
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
saved_cfg = ckpt.get('config')
cfg = JazzConfig(**saved_cfg) if isinstance(saved_cfg, dict) else saved_cfg
model = JazzLM(cfg)
model.load_state_dict(ckpt['model'], strict=False)
if tok.vocab_size != cfg.vocab_size:
    model.resize_token_embeddings(tok.vocab_size)
model.to(device).eval()
print(f'Loaded Jazz AI V1.0: {model.num_parameters()/1e6:.0f}M params on {device}')

def chat(prompt, system='You are Jazz AI V1.0, a helpful Hindi/Hinglish assistant.', max_new=200, temp=0.7, top_k=50):
    messages = [{'role':'system','content':system}, {'role':'user','content':prompt}]
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_assistant_mask=False)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    n_in = x.size(1)
    with torch.no_grad():
        out = model.generate(x, max_new_tokens=max_new, temperature=temp, top_k=top_k, eos_token_id=tok.eos_token_id)
    new_ids = out[0, n_in:].tolist()
    im_end = tok.eos_token_id
    if im_end in new_ids: new_ids = new_ids[:new_ids.index(im_end)]
    return tok.decode(new_ids).strip()

print('\n--- Test ---')
print('Q: Tum kaun ho?')
print('A:', chat('Tum kaun ho?'))

In [ ]:
# Interactive multi-turn chat
history = [{'role':'system','content':'You are Jazz AI V1.0, a helpful Hindi/Hinglish assistant. Direct aur short jawaab do.'}]

while True:
    user_input = input('You: ').strip()
    if not user_input or user_input.lower() in ['/quit', 'exit']: break
    if user_input == '/reset':
        history = [history[0]]
        print('(history reset)'); continue
    history.append({'role':'user','content':user_input})
    ids = tok.apply_chat_template(history, add_generation_prompt=True, return_assistant_mask=False)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    n_in = x.size(1)
    with torch.no_grad():
        out = model.generate(x, max_new_tokens=200, temperature=0.7, top_k=50, eos_token_id=tok.eos_token_id)
    new_ids = out[0, n_in:].tolist()
    im_end = tok.eos_token_id
    if im_end in new_ids: new_ids = new_ids[:new_ids.index(im_end)]
    reply = tok.decode(new_ids).strip()
    print(f'Jazz AI: {reply}\n')
    history.append({'role':'assistant','content':reply})

## Download Final Checkpoint
Run this to download the trained model to your local machine.

In [ ]:
import shutil, os
archive = shutil.make_archive('/kaggle/working/jazz_ai_v1', 'zip', '.', 'out')
print('Created archive:', archive)
print('Download it from the Kaggle Output panel after the run.')
